# new_pipeline smoke test

This notebook assumes these files exist:

- `data/transcriptomics_filtered.joblib`
- `data/proteomics_matched.joblib`
- `data/protein_list.txt`

Outputs go to `data/results/`.


In [2]:

from pathlib import Path
from joblib import load
import sys

ROOT = Path('.').resolve()
DATA_DIR = ROOT / 'data'
SRC_DIR = ROOT / 'src'
RESULTS_DIR = DATA_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

transcript_path = DATA_DIR / 'transcriptomics_filtered.joblib'
protein_path = DATA_DIR / 'proteomics_matched.joblib'
protein_list_path = DATA_DIR / 'protein_list.txt'

print('DATA_DIR:', DATA_DIR)
print('Exists transcriptomics:', transcript_path.exists(), transcript_path)
print('Exists proteomics:', protein_path.exists(), protein_path)
print('Exists protein_list:', protein_list_path.exists(), protein_list_path)

assert transcript_path.exists(), 'Missing transcriptomics_filtered.joblib in data/'
assert protein_path.exists(), 'Missing proteomics_matched.joblib in data/'
assert protein_list_path.exists(), 'Missing protein_list.txt in data/'


DATA_DIR: C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\03. elasticNet\new_pipeline\data
Exists transcriptomics: True C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\03. elasticNet\new_pipeline\data\transcriptomics_filtered.joblib
Exists proteomics: True C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\03. elasticNet\new_pipeline\data\proteomics_matched.joblib
Exists protein_list: True C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\03. elasticNet\new_pipeline\data\protein_list.txt


In [3]:

X = load(transcript_path)
Y = load(protein_path)

proteins = [p.strip() for p in protein_list_path.read_text().splitlines()
            if p.strip() and not p.strip().startswith('#')]
print('Proteins in list:', len(proteins))
test_proteins = proteins[:3]
print('Testing proteins:', test_proteins)


Proteins in list: 20
Testing proteins: ['PPP1CC', 'DDX1', 'PSMC1']


In [4]:

sys.path.insert(0, str(SRC_DIR))
from new_functions import run_protein_group

param_grid = {"alpha": [0.1, 0.5, 1.0], "l1_ratio": [0.1, 0.5, 0.9]}

out_best = run_protein_group(
    proteins=test_proteins,
    pipeline="fixed_percentile",
    X=X,
    Y=Y,
    param_grid=param_grid,
    percentile=5,
    n_splits=3,
    save_only_best=True,
    output_dir=str(RESULTS_DIR),
    zscore=True,
    fit_intercept=False,
)
out_best


,protein,best_sample,best_abs_error,mse_all_others,params_dict,feat_coeffs_dict
0,PPP1CC,EOL1_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,0.001884,0.066236,"{'alpha': 0.5, 'l1_ratio': 0.1}","{'ABCB4': -0.0, 'ABHD14B': -0.0, 'ABHD17B': -0..."
1,DDX1,TE6_OESOPHAGUS,0.000637,0.072147,"{'alpha': 0.1, 'l1_ratio': 0.5}","{'AAAS': 0.0, 'AARS2': 0.0, 'AASDHPPT': 0.0, '..."
2,PSMC1,NCIH1944_LUNG,0.004271,0.068504,"{'alpha': 0.5, 'l1_ratio': 0.1}","{'AASDHPPT': 0.0, 'ABCC3': -0.0, 'ABCD2': 0.0,..."


In [5]:

# Verify per-protein files were written
for p in test_proteins:
    fpq = RESULTS_DIR / f"{p}_best.parquet"
    fcsv = RESULTS_DIR / f"{p}_best.csv"
    print(p, "parquet?", fpq.exists(), "csv?", fcsv.exists())


PPP1CC parquet? True csv? False
DDX1 parquet? True csv? False
PSMC1 parquet? True csv? False


In [6]:
# Run the CLI script for a small batch using target_chunks (dynamic group_size)
import subprocess, shlex

cmd = [
    "python", str(SRC_DIR / "new_pipeline.py"),
    "--start_idx", "0",
    "--target_chunks", "10",          # dynamic group_size = ceil(total_proteins/10)
    "--pipeline", "fixed_percentile",
    "--percentile", "5",
    "--n_splits", "5",
    "--data_dir", str(DATA_DIR),
    "--results_dir", str(RESULTS_DIR),
    "--protein_list", str(protein_list_path),
]
print("CMD:", " ".join(shlex.quote(c) for c in cmd))
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout)
print(r.stderr)


CMD: python 'C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\03. elasticNet\new_pipeline\src\new_pipeline.py' --start_idx 0 --target_chunks 10 --pipeline fixed_percentile --percentile 5 --n_splits 5 --data_dir 'C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\03. elasticNet\new_pipeline\data' --results_dir 'C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\03. elasticNet\new_pipeline\data\results' --protein_list 'C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\03. elasticNet\new_pipeline\data\protein_list.txt'
Loading data...
Selected proteins: 0..1 (count=2, total_list=20)
Pipeline=fixed_percentile, save_only_best=True, results_dir=C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\03. elasticNet\new_pipeline\data\results
group_size=2 (arg_group_size=None, target_chunks=10)
Saved batch results to: C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\03. elasticNet\new_pipeline\data\results\batch_0_1_fixed_percentile.parquet
Done.




In [8]:
# Inspect the saved batch parquet (batch_0_1_fixed_percentile.parquet)

from pathlib import Path
import pandas as pd

batch_path = Path(RESULTS_DIR) / "batch_0_1_fixed_percentile.parquet"
print("Parquet path:", batch_path)
print("Exists:", batch_path.exists())

df = pd.read_parquet(batch_path)

print("\nShape:", df.shape)
print("\nColumns:", list(df.columns))

# Show the whole dataframe (small)
display(df)

# Optional: quick peek at the dict fields without dumping huge text
if "params_dict" in df.columns:
    print("\nparams_dict (first row):", df.loc[0, "params_dict"])
if "feat_coeffs_dict" in df.columns:
    d0 = df.loc[0, "feat_coeffs_dict"]
    print("\n#features in feat_coeffs_dict (first row):", len(d0) if isinstance(d0, dict) else type(d0))


Parquet path: C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\03. elasticNet\new_pipeline\data\results\batch_0_1_fixed_percentile.parquet
Exists: True

Shape: (2, 6)

Columns: ['protein', 'best_sample', 'best_abs_error', 'mse_all_others', 'params_dict', 'feat_coeffs_dict']


,protein,best_sample,best_abs_error,mse_all_others,params_dict,feat_coeffs_dict
0,PPP1CC,SW837_LARGE_INTESTINE,0.000546,0.025581,"{'alpha': 0.1, 'l1_ratio': 0.1}","{'AAAS': None, 'AARS2': None, 'AASDHPPT': None..."
1,DDX1,TE6_OESOPHAGUS,0.000637,0.072147,"{'alpha': 0.1, 'l1_ratio': 0.5}","{'AAAS': 0.0, 'AARS2': 0.0, 'AASDHPPT': 0.0, '..."



params_dict (first row): {'alpha': 0.1, 'l1_ratio': 0.1}

#features in feat_coeffs_dict (first row): 2233
